In [ ]:
!pip install transformers trl

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer
import torch

In [2]:
model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"
# model_name = "meta-llama/Llama-2-7b-hf"

model = AutoModelForCausalLM.from_pretrained(model_name,
                                            #  torch_dtype=torch.float16
                                             )
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          # torch_dtype=torch.float16
                                          )
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32256, 2048)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5504, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5504, bias=False)
          (down_proj): Linear(in_features=5504, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-06)
    (r

In [3]:
input_text = "What should I do on a trip to Europe?"

input_ids = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**input_ids, max_length=128)
print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


<｜begin▁of▁sentence｜>What should I do on a trip to Europe?


1. Visit museums: Many of Europe's top-rated museums offer free access to their collections.

2. Visit theatres: Many of Europe's top-rated theatres offer free access to their shows.

3. Visit the parks: Europe has a rich ecosystem of parks. Visit one of the largest in Europe.

4. Visit the sea: Europe has a rich ecosystem of seashores. Visit one of the largest in Europe.

5. Visit the city: Europe has a rich ecosystem of


In [4]:
from datasets import load_dataset

dataset_name = "TrainDataset/Pyranet_LoRa_256_ins_code.jsonl"
dataset = load_dataset('json', data_files=dataset_name, split="train[0:1000]")

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'code'],
    num_rows: 1000
})

In [5]:
print(f"Instruction is: {dataset[0]['instruction']}")
print(f"Response is: {dataset[0]['code']}")
dataset

Instruction is: The Verilog code defines a module named `VCC` that outputs a constant high logic level (1). The output `V` is always set to 1.

module VCC (output V);
Response is: module VCC (output V);
   assign V = 1'b1;
endmodule


Dataset({
    features: ['instruction', 'code'],
    num_rows: 1000
})

In [23]:
def formatting_prompts_func(example):
    # output_texts = []
    instructions = example['instruction']
    codes = example['code']
    # print("here ", example)
    # for instruction, code in zip(instructions, codes):
    # for i in range(len(example['instruction'])):
    #     if example['instruction'][i] in ['open_qa', 'general_qa']:
    #       text = f"Instruction:\n{example['instruction']}\n\nResponse:\n{example['response']}"
    #       output_texts.append(text)
    text = f"""You are an AI programming assistant, utilizing the DeepSeek Coder model, developed by DeepSeek Company, and you only answer questions related to computer science. For politically sensitive questions, security and privacy issues, and other non-computer science questions, you will refuse to answer.
### Instruction:
{instructions}
### Response:
{codes}
<|EOT|>"""
    # output_texts.append(text)
    return text


In [24]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
)
print("Initialized trainer for training!")

Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Initialized trainer for training!


In [ ]:
trainer.train()